In [4]:
import pandas as pd

print("Cargando archivos...")
dim_category = pd.read_csv("DIM_CATEGORY.csv")
dim_product = pd.read_excel("DIM_PRODUCT.xlsx", sheet_name="Sheet1")
dim_segment = pd.read_excel("DIM_SEGMENT.xlsx", sheet_name="SEGMENT")
dim_calendar = pd.read_excel("DIM_CALENDAR.xlsx", sheet_name="Sheet1")
fact_sales = pd.read_csv("FACT_SALES.csv")

print("\n--- Estructura inicial ---")
print("FACT_SALES info:")
print(fact_sales.info())
print("\nPrimeras filas de FACT_SALES:")
print(fact_sales.head())

dim_category["CATEGORY"] = (
    dim_category["CATEGORY"].str.strip().str.replace("\r\n", "")
)
dim_product["ITEM"] = dim_product["ITEM"].astype(str).str.strip()
fact_sales["ITEM_CODE"] = fact_sales["ITEM_CODE"].astype(str).str.strip()

initial_rows = len(fact_sales)
fact_sales = fact_sales.drop_duplicates()
print(
    f"\nSe eliminaron {initial_rows - len(fact_sales)} filas duplicadas en FACT_SALES."
)

print("\nValores nulos por columna en FACT_SALES:")
print(fact_sales.isnull().sum())
print("\nValores nulos por columna en DIM_PRODUCT:")
print(dim_product.isnull().sum())

dim_product["ATTR3"] = dim_product["ATTR3"].fillna("NO DEFINIDO")

consolidated = fact_sales.merge(
    dim_product, left_on="ITEM_CODE", right_on="ITEM", how="left"
)

consolidated = consolidated.merge(dim_calendar, on="WEEK", how="left")

consolidated = consolidated.merge(
    dim_category, left_on="CATEGORY", right_on="ID_CATEGORY", how="left"
)

consolidated["DATE"] = pd.to_datetime(consolidated["DATE"])

consolidated.to_csv("CONSOLIDATED_SALES.csv", index=False)
print(
    "\n¡Proceso finalizado con éxito! El conjunto consolidado se guardó como 'CONSOLIDATED_SALES.csv'."
)

Cargando archivos...

--- Estructura inicial ---
FACT_SALES info:
<class 'pandas.DataFrame'>
RangeIndex: 122002 entries, 0 to 122001
Data columns (total 6 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   WEEK                         122002 non-null  str    
 1   ITEM_CODE                    122002 non-null  str    
 2   TOTAL_UNIT_SALES             122002 non-null  float64
 3   TOTAL_VALUE_SALES            122002 non-null  float64
 4   TOTAL_UNIT_AVG_WEEKLY_SALES  122002 non-null  float64
 5   REGION                       122002 non-null  str    
dtypes: float64(3), str(3)
memory usage: 10.0 MB
None

Primeras filas de FACT_SALES:
    WEEK         ITEM_CODE  TOTAL_UNIT_SALES  TOTAL_VALUE_SALES  \
0  34-22  7501058792808BP2             0.006              0.139   
1  34-22     7501058715883             0.487            116.519   
2  34-22     7702626213774             1.391             68.453   
3  34-22  